# 06 — Explicabilité, fidélité des explications et analyse des biais

**Principe** : la décision vient du modèle entraîné et évalué (notebook 05). Ce notebook **explique** cette décision et **vérifie** que les explications sont fidèles ; il ne modifie jamais la probabilité.

| Section | Question | Méthode |
|---|---|---|
| 1 | Sur quelle modalité le modèle s'appuie-t-il ? | importance par permutation (par modalité et par variable) |
| 2 | Pourquoi ce patient ? | contribution par remplacement (équivalente à SHAP pour un modèle linéaire) |
| 3 | Où regarde le modèle dans l'image ? | **carte d'occlusion** : on masque chaque zone et on mesure la baisse de probabilité |
| 4 | Les cartes sont-elles fidèles ? | test de suppression : masquer les zones désignées doit faire baisser la probabilité plus que masquer des zones au hasard |
| 5 | Le modèle utilise-t-il l'encre chirurgicale ? | détection des marques d'encre + performance et attention sur ces images |
| 6 | Que dire au médecin ? | rapport texte par **gabarit** à partir des sorties calculées |
| 7–8 | Où se trompe-t-il ? Pour qui ? | analyse d'erreurs, analyse par phototype |

*Pourquoi l'occlusion plutôt que Grad-CAM ?* Elle traverse **tout** le modèle final (image → concepts → fusion → probabilité) sans hypothèse sur son architecture et mesure un effet réel sur la décision : elle est fidèle par construction.

**Prérequis** : notebook 05 exécuté (`outputs/checkpoints/final_model.joblib`). DermLIP est rechargé (cache Hugging Face) pour les cartes d'occlusion. Sur CPU, les sections 3 à 5 prennent de l'ordre de 10 à 20 minutes.

## 0. Imports, données, fonctions et modèles communs

In [ ]:
from pathlib import Path
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, f1_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = ROOT / "zr7vgbcyr2-1" / "metadata.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
EXTERNAL_DIR = ROOT / "data" / "external" / "derm1m"
CONFIG_DIR = ROOT / "configs"
FIG_DIR = ROOT / "outputs" / "figures"
REPORT_DIR = ROOT / "outputs" / "reports"
CKPT_DIR = ROOT / "outputs" / "checkpoints"
for d in (PROCESSED_DIR, FIG_DIR, REPORT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# variables cliniques retenues (notebook 01) — le profil patient n'est utilisé que dans l'expérience E8
NUM_COLS = ["age"]
BOOL_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
CAT_COLS = ["region"]
FEATURES = NUM_COLS + BOOL_COLS + CAT_COLS
PROFILE_NUM = ["fitspatrick", "diameter_1", "diameter_2"]
PROFILE_BOOL = ["smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history"]
PROFILE_CAT = ["gender"]
PROFILE_COLS = PROFILE_NUM + PROFILE_BOOL + PROFILE_CAT

CANCER_CLASSES = ["BCC", "SCC", "MEL"]
DERMLIP_ID = "hf-hub:redlessone/DermLIP_ViT-B-16"

In [ ]:
splits = pd.read_csv(PROCESSED_DIR / "splits.csv")
meta = pd.read_csv(CSV_PATH, na_values=["UNK"])

df = splits.merge(meta[["img_id"] + FEATURES + PROFILE_COLS], on="img_id", how="left",
                  validate="one_to_one")
TO_FLOAT = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
for c in BOOL_COLS + PROFILE_BOOL:                     # True / False / NaN -> 1.0 / 0.0 / NaN
    df[c] = df[c].map(TO_FLOAT)

dev = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
y_dev, folds, groups_dev = dev["label"].to_numpy(), dev["fold"].to_numpy(), dev["patient_id"].to_numpy()
y_test, groups_test = test["label"].to_numpy(), test["patient_id"].to_numpy()

print(f"Développement : {len(dev)} images, {dev['patient_id'].nunique()} patients | "
      f"test : {len(test)} images (non utilisé avant l'évaluation finale)")
assert set(folds) == set(range(N_FOLDS))

In [ ]:
def metrics_at(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "Exactitude équilibrée": balanced_accuracy_score(y_true, y_pred),
        "Sensibilité": recall_score(y_true, y_pred, pos_label=1),
        "Spécificité": recall_score(y_true, y_pred, pos_label=0),
        "F1": f1_score(y_true, y_pred),
    }


def take_rows(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


def fit_model(model, X, y, groups=None):
    """Entraîne un modèle ; transmet `groups` aux modèles qui choisissent leurs hyperparamètres par patient."""
    needs_groups = isinstance(model, GridSearchCV) or getattr(model, "uses_groups", False)
    if needs_groups:
        assert groups is not None, "ce modèle a besoin des identifiants patients (groups)"
        return model.fit(X, y, groups=groups)
    return model.fit(X, y)


def cross_validate(make_model, X, y, folds, groups=None, return_models=False):
    """Validation croisée sur les 5 plis groupés par patient : métriques par pli + prédictions hors pli."""
    oof, rows, models = np.zeros(len(y)), [], []
    for k in range(N_FOLDS):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        model = make_model()
        fit_model(model, take_rows(X, tr), y[tr], None if groups is None else groups[tr])
        oof[va] = model.predict_proba(take_rows(X, va))[:, 1]
        rows.append({"pli": k, **metrics_at(y[va], oof[va])})
        models.append(model)
    per_fold = pd.DataFrame(rows)
    return (per_fold, oof, models) if return_models else (per_fold, oof)


def summarize(name, per_fold):
    m = per_fold.drop(columns="pli")
    return {"Expérience": name, **{c: f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m.columns},
            "AUC moyenne": m["AUC"].mean()}


def grouped_bootstrap(y, p, groups, p_ref=None, n_boot=2000, seed=SEED):
    """AUC et IC 95 % par bootstrap des patients ; avec `p_ref`, différence d'AUC appariée."""
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    rng = np.random.default_rng(seed)
    aucs, diffs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
        if y[idx].min() == y[idx].max():
            continue
        a = roc_auc_score(y[idx], p[idx])
        aucs.append(a)
        if p_ref is not None:
            diffs.append(a - roc_auc_score(y[idx], np.asarray(p_ref)[idx]))
    out = {"AUC": roc_auc_score(y, p), "IC95 bas": np.percentile(aucs, 2.5), "IC95 haut": np.percentile(aucs, 97.5)}
    if p_ref is not None:
        d = np.asarray(diffs)
        out.update({"ΔAUC": out["AUC"] - roc_auc_score(y, p_ref),
                    "ΔAUC IC95 bas": np.percentile(d, 2.5), "ΔAUC IC95 haut": np.percentile(d, 97.5),
                    "P(Δ ≤ 0)": float((d <= 0).mean())})
    return out


def make_clinical_preprocessor(num=NUM_COLS, bools=BOOL_COLS, cats=CAT_COLS):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("bool", SimpleImputer(strategy="most_frequent"), bools),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cats),
    ])


C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]


def make_probe(c_grid=C_GRID):
    """Sonde linéaire : standardisation + logistique, C choisi par validation croisée groupée (patients)."""
    pipe = Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    return GridSearchCV(pipe, {"clf__C": c_grid}, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def load_features(name):
    """Charge un cache de caractéristiques et renvoie une fonction frame -> matrice alignée sur frame['img_id']."""
    feats = np.load(PROCESSED_DIR / f"features_{name}.npy")
    ids = pd.read_csv(PROCESSED_DIR / f"features_{name}_ids.csv")["img_id"]
    index = pd.Series(np.arange(len(ids)), index=ids.to_numpy())
    return lambda frame: feats[index.loc[frame["img_id"]].to_numpy()]


results, oof_preds = [], {}

In [ ]:
from functools import partial

import joblib
import torch
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit


def make_linear_fusion(clin=True, img=False, cpt=False, profile=False):
    """Fusion précoce : blocs (clinique / image / concepts) prétraités puis une régression logistique.
    C (et la réduction PCA de l'image) sont choisis par validation croisée groupée par patient."""
    blocks = []
    if clin:
        num, bools, cats = ((NUM_COLS + PROFILE_NUM, BOOL_COLS + PROFILE_BOOL, CAT_COLS + PROFILE_CAT)
                            if profile else (NUM_COLS, BOOL_COLS, CAT_COLS))
        blocks.append(("clin", make_clinical_preprocessor(num, bools, cats), num + bools + cats))
    if img:
        blocks.append(("img", Pipeline([("scale", StandardScaler()), ("pca", PCA(random_state=SEED))]), IMG_COLS))
    if cpt:
        blocks.append(("cpt", StandardScaler(), CONCEPT_COLS))
    pipe = Pipeline([("prep", ColumnTransformer(blocks)), ("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    grid = {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]}
    if img:
        grid["prep__img__pca__n_components"] = [16, 64, 128]
    return GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


class LateFusion(BaseEstimator, ClassifierMixin):
    """Fusion tardive (stacking) : un modèle par modalité, combinés par une logistique apprise
    sur leurs prédictions hors pli internes (groupées par patient)."""
    uses_groups = True

    def __init__(self, base_factories=(), n_inner=3):
        self.base_factories = base_factories
        self.n_inner = n_inner

    def fit(self, X, y, groups):
        y, groups = np.asarray(y), np.asarray(groups)
        Z = np.zeros((len(y), len(self.base_factories)))
        for tr, va in GroupKFold(n_splits=self.n_inner).split(X, y, groups):
            for j, (_, factory) in enumerate(self.base_factories):
                m = fit_model(factory(), take_rows(X, tr), y[tr], groups[tr])
                Z[va, j] = _logit(m.predict_proba(take_rows(X, va))[:, 1])
        self.meta_ = LogisticRegression(class_weight="balanced").fit(Z, y)
        self.bases_ = [fit_model(f(), X, y, groups) for _, f in self.base_factories]
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        Z = np.column_stack([_logit(m.predict_proba(X)[:, 1]) for m in self.bases_])
        return self.meta_.predict_proba(Z)


class _GatedNet(torch.nn.Module):
    def __init__(self, dims, hidden, dropout):
        super().__init__()
        self.encoders = torch.nn.ModuleList([
            torch.nn.Sequential(torch.nn.Linear(d, hidden), torch.nn.GELU(), torch.nn.Dropout(dropout)) for d in dims])
        self.gate = torch.nn.Linear(hidden * len(dims), len(dims))
        self.head = torch.nn.Sequential(torch.nn.LayerNorm(hidden), torch.nn.Linear(hidden, 1))

    def forward(self, xs):
        hs = [enc(x) for enc, x in zip(self.encoders, xs)]
        w = torch.softmax(self.gate(torch.cat(hs, dim=1)), dim=1)        # poids de chaque modalité
        fused = sum(w[:, i:i + 1] * h for i, h in enumerate(hs))
        return self.head(fused).squeeze(1), w


class GatedFusion(BaseEstimator, ClassifierMixin):
    """Fusion à porte : un petit réseau par modalité, une porte apprend le poids de chaque modalité
    pour chaque patient. Arrêt anticipé sur 15 % des patients du pli ; moyenne de plusieurs graines."""
    uses_groups = True
    MODALITIES = ("clinique", "image", "concepts")

    def __init__(self, hidden=32, dropout=0.3, lr=1e-3, weight_decay=1e-2, max_epochs=400,
                 patience=30, n_seeds=5, val_size=0.15, img_pca=64):
        self.hidden, self.dropout, self.lr, self.weight_decay = hidden, dropout, lr, weight_decay
        self.max_epochs, self.patience, self.n_seeds = max_epochs, patience, n_seeds
        self.val_size, self.img_pca = val_size, img_pca

    def _blocks(self):
        return [("clin", make_clinical_preprocessor(), FEATURES),
                ("img", Pipeline([("scale", StandardScaler()),
                                  ("pca", PCA(n_components=self.img_pca, random_state=SEED))]), IMG_COLS),
                ("cpt", StandardScaler(), CONCEPT_COLS)]

    def _transform(self, X):
        return [torch.tensor(np.asarray(p.transform(X[cols]), dtype=np.float32))
                for (_, _, cols), p in zip(self._blocks(), self.preps_)]

    def fit(self, X, y, groups):
        y = np.asarray(y, dtype=np.float32)
        tr, va = next(GroupShuffleSplit(n_splits=1, test_size=self.val_size, random_state=SEED)
                      .split(X, y, groups))
        Xtr, Xva = take_rows(X, tr), take_rows(X, va)
        self.preps_ = [p.fit(Xtr[cols]) for _, p, cols in self._blocks()]
        xtr, xva = self._transform(Xtr), self._transform(Xva)
        ytr, yva = torch.tensor(y[tr]), torch.tensor(y[va])
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - y[tr]).sum() / max(y[tr].sum(), 1.0)))
        self.nets_ = []
        for s in range(self.n_seeds):
            torch.manual_seed(SEED + s)
            net = _GatedNet([x.shape[1] for x in xtr], self.hidden, self.dropout)
            opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            best, best_state, wait = np.inf, None, 0
            for _ in range(self.max_epochs):
                net.train()
                opt.zero_grad()
                loss_fn(net(xtr)[0], ytr).backward()
                opt.step()
                net.eval()
                with torch.no_grad():
                    v = loss_fn(net(xva)[0], yva).item()
                if v < best - 1e-4:
                    best, wait = v, 0
                    best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
                else:
                    wait += 1
                    if wait >= self.patience:
                        break
            net.load_state_dict(best_state)
            net.eval()
            self.nets_.append(net)
        self.classes_ = np.array([0, 1])
        return self

    def _forward(self, X):
        xs = self._transform(X)
        with torch.no_grad():
            outs = [net(xs) for net in self.nets_]
        p = np.mean([torch.sigmoid(o[0]).numpy() for o in outs], axis=0)
        w = np.mean([o[1].numpy() for o in outs], axis=0)
        return p, w

    def predict_proba(self, X):
        p, _ = self._forward(X)
        return np.column_stack([1 - p, p])

    def modality_weights(self, X):
        return pd.DataFrame(self._forward(X)[1], columns=list(self.MODALITIES))


def expected_calibration_error(y, p, n_bins=10):
    y, p = np.asarray(y), np.asarray(p)
    idx = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    rows, ece = [], 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
            rows.append({"confiance moyenne": p[m].mean(), "fréquence observée": y[m].mean(), "n": int(m.sum())})
    return ece, pd.DataFrame(rows)

In [ ]:
take_img = load_features("dermlip_vitb16")
emb_bank = np.load(PROCESSED_DIR / "concept_text_embeddings.npz", allow_pickle=True)
selected = pd.read_csv(PROCESSED_DIR / "concepts_selected.csv")
CONCEPT_NAMES = selected.loc[selected["retenu"], "concept"].tolist()
_name_to_row = {n: i for i, n in enumerate(emb_bank["names"])}
CPT_EMB = emb_bank["concept_emb"][[_name_to_row[n] for n in CONCEPT_NAMES]]
REF_EMB = emb_bank["ref_emb"]

N_IMG = take_img(dev.head(1)).shape[1]
IMG_COLS = [f"img_{i}" for i in range(N_IMG)]
CONCEPT_COLS = [f"c_{n}" for n in CONCEPT_NAMES]


def concept_scores(emb):
    """Score de concept = similarité image/concept moins similarité image/référence neutre."""
    return emb @ CPT_EMB.T - (emb @ REF_EMB)[:, None]


def build_frame(frame, emb=None):
    """Table complète : variables cliniques + plongement image DermLIP + scores de concepts."""
    emb = take_img(frame) if emb is None else emb
    clin = frame[FEATURES + PROFILE_COLS].reset_index(drop=True)
    img = pd.DataFrame(emb, columns=IMG_COLS)
    cpt = pd.DataFrame(concept_scores(emb), columns=CONCEPT_COLS)
    return pd.concat([clin, img, cpt], axis=1)


print("Concepts retenus :", len(CONCEPT_NAMES), "| dimension image :", N_IMG)

## Chargement du modèle final et contrôle de reproductibilité

In [ ]:
bundle = joblib.load(CKPT_DIR / "final_model.joblib")
final_model = bundle["modele"]
assert bundle["concepts"] == CONCEPT_NAMES, "concepts différents de ceux du notebook 05"
T_MAIN = bundle["seuils"][bundle["seuil_principal"]]

X_test = build_frame(test)
p_test = final_model.predict_proba(X_test)[:, 1]
saved = pd.read_csv(REPORT_DIR / "05_predictions_test.csv").set_index("img_id").loc[test["img_id"], "proba"]
assert np.allclose(p_test, saved.to_numpy(), atol=1e-6), "les prédictions ne correspondent pas au notebook 05"
print(f"Modèle : {bundle['nom']} | seuil de triage : {T_MAIN:.3f} | AUC test : {roc_auc_score(y_test, p_test):.3f}")

LABELS_FR = dict(zip(emb_bank["names"], emb_bank["labels_fr"]))
CLIN_FR = {"age": "âge", "itch": "démangeaison", "grew": "a grossi", "hurt": "douleur",
           "changed": "a changé", "bleed": "saignement", "elevation": "relief", "region": "localisation"}

## 1. Importance globale par permutation (sur le test)

In [ ]:
def permutation_drop(model, X, y, col_groups, n_repeats=20, seed=SEED):
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y, model.predict_proba(X)[:, 1])
    rows = []
    for name, cols in col_groups.items():
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].to_numpy()[rng.permutation(len(X))]
            drops.append(base - roc_auc_score(y, model.predict_proba(Xp)[:, 1]))
        rows.append({"élément": name, "baisse d'AUC": np.mean(drops), "écart-type": np.std(drops)})
    return pd.DataFrame(rows).sort_values("baisse d'AUC", ascending=False)


modalities = permutation_drop(final_model, X_test, y_test, {
    "clinique": FEATURES, "image (plongement)": IMG_COLS, "concepts": CONCEPT_COLS,
    "image + concepts": IMG_COLS + CONCEPT_COLS})
variables = permutation_drop(final_model, X_test, y_test,
                             {**{CLIN_FR[c]: [c] for c in FEATURES},
                              **{f"concept : {LABELS_FR[n]}": [c] for n, c in zip(CONCEPT_NAMES, CONCEPT_COLS)}},
                             n_repeats=10)

fig, axes = plt.subplots(1, 2, figsize=(15, 0.32 * len(variables) + 2))
axes[0].barh(modalities["élément"], modalities["baisse d'AUC"], xerr=modalities["écart-type"], color="#4C72B0")
axes[0].invert_yaxis()
axes[0].set(title="Importance des modalités", xlabel="baisse d'AUC quand la modalité est brouillée")
axes[1].barh(variables["élément"], variables["baisse d'AUC"], xerr=variables["écart-type"], color="#DD8452")
axes[1].invert_yaxis()
axes[1].set(title="Importance des variables et concepts", xlabel="baisse d'AUC")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_importance_permutation.png", dpi=150)
plt.show()
modalities.round(4)

## 2. Explication locale : contribution par remplacement

Pour un patient, on remplace **un seul** élément par une valeur de référence (médiane ou valeur la plus fréquente du développement) et on mesure la variation du logit de la probabilité. Pour un modèle linéaire, c'est exactement la décomposition de SHAP ; pour un autre modèle, c'est une approximation locale honnête et facile à expliquer.

In [ ]:
REF_CLIN, REF_CPT, REF_IMG = bundle["reference_clinique"], bundle["reference_concepts"], bundle["reference_image"]


def logit_of(X):
    return _logit(final_model.predict_proba(X)[:, 1])


def local_contributions(xrow):
    """xrow : DataFrame d'une ligne (issu de build_frame). Contribution > 0 : pousse vers « cancer »."""
    variants, meta_rows = [], []
    for c in FEATURES:
        v = xrow.copy()
        v[c] = REF_CLIN[c]
        variants.append(v)
        meta_rows.append(("clinique", c, CLIN_FR[c]))
    for n, c in zip(CONCEPT_NAMES, CONCEPT_COLS):
        v = xrow.copy()
        v[c] = REF_CPT[c]
        variants.append(v)
        meta_rows.append(("concept", n, f"concept « {LABELS_FR[n]} »"))
    v = xrow.copy()
    v.loc[:, IMG_COLS] = REF_IMG.reshape(1, -1)
    variants.append(v)
    meta_rows.append(("image", "image", "aspect global de l'image"))
    delta = logit_of(xrow)[0] - logit_of(pd.concat(variants, ignore_index=True))
    out = pd.DataFrame(meta_rows, columns=["type", "clé", "élément"]).assign(contribution=delta)
    return out.reindex(out["contribution"].abs().sort_values(ascending=False).index).reset_index(drop=True)


pos = test.index[(y_test == 1) & (p_test >= T_MAIN)]
example = int(pos[np.argmax(p_test[pos])]) if len(pos) else int(np.argmax(p_test))
print(f"Exemple : {test.loc[example, 'img_id']} ({test.loc[example, 'diagnostic']}), probabilité {p_test[example]:.3f}")
local_contributions(X_test.iloc[[example]]).head(10).round(3)

## 3. Cartes d'occlusion

L'image prétraitée (224 × 224) est découpée en 8 × 8 zones. Chaque zone est remplacée à son tour par la couleur moyenne (0 après normalisation), l'image est ré-encodée par DermLIP, les concepts sont recalculés et le modèle final redonne une probabilité. **Rouge** : la zone pousse vers « cancer » ; **bleu** : vers « non-cancer ».

> La cellule suivante charge DermLIP (cache Hugging Face, téléchargé au notebook 03).

In [ ]:
try:
    import open_clip
    import torch
except ImportError as e:
    raise ImportError("Installez open_clip :  pip install open_clip_torch") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
model, _, preprocess = open_clip.create_model_and_transforms(DERMLIP_ID)
tokenizer = open_clip.get_tokenizer(DERMLIP_ID)
model = model.to(device).eval()
LOGIT_SCALE = float(model.logit_scale.exp().item())
print("DermLIP chargé sur", device, "| échelle des logits :", round(LOGIT_SCALE, 1))


@torch.no_grad()
def encode_texts(texts):
    emb = model.encode_text(tokenizer(texts).to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def prompt_embedding(prompts):
    """Moyenne des plongements de plusieurs formulations, renormalisée (« ensemble de prompts »)."""
    e = encode_texts(prompts).mean(axis=0)
    return e / np.linalg.norm(e)


@torch.no_grad()
def encode_tensors(x):
    emb = model.encode_image(x.to(device)).float()
    return torch.nn.functional.normalize(emb, dim=-1).cpu().numpy()


def load_tensor(path):
    with Image.open(path) as im:
        return preprocess(im.convert("RGB"))

In [ ]:
from matplotlib.colors import rgb_to_hsv
from torchvision import transforms as T

GRID = 8
NORM = next(t for t in preprocess.transforms if isinstance(t, T.Normalize))
MEAN, STD = torch.tensor(NORM.mean).view(3, 1, 1), torch.tensor(NORM.std).view(3, 1, 1)


def to_rgb(x):
    return (x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()


def cell_slices(i, j, size):
    step = size // GRID
    return slice(i * step, (i + 1) * step), slice(j * step, (j + 1) * step)


def proba_of_tensors(i_test, tensors, batch=16):
    embs = np.concatenate([encode_tensors(torch.stack(tensors[s:s + batch])) for s in range(0, len(tensors), batch)])
    rows = pd.concat([test.iloc[[i_test]]] * len(embs), ignore_index=True)
    return final_model.predict_proba(build_frame(rows, emb=embs))[:, 1]


heat_cache = {}


def occlusion_map(i_test):
    if i_test in heat_cache:
        return heat_cache[i_test]
    x = load_tensor(ROOT / test.loc[i_test, "img_path"])
    variants = [x]
    for i in range(GRID):
        for j in range(GRID):
            v = x.clone()
            si, sj = cell_slices(i, j, x.shape[1])
            v[:, si, sj] = 0.0
            variants.append(v)
    p = proba_of_tensors(i_test, variants)
    heat_cache[i_test] = (x, p[0], (p[0] - p[1:]).reshape(GRID, GRID))
    return heat_cache[i_test]


def show_occlusion(ax_img, ax_heat, i_test, title=""):
    x, p0, heat = occlusion_map(i_test)
    rgb = to_rgb(x)
    ax_img.imshow(rgb)
    ax_img.set_title(title, fontsize=9)
    vmax = max(np.abs(heat).max(), 1e-6)
    ax_heat.imshow(rgb)
    ax_heat.imshow(np.kron(heat, np.ones((x.shape[1] // GRID, x.shape[2] // GRID))), cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, alpha=0.5)
    ax_heat.set_title(f"occlusion (p = {p0:.2f})", fontsize=9)
    for a in (ax_img, ax_heat):
        a.axis("off")


x0, p0, _ = occlusion_map(example)
print(f"Contrôle : probabilité recalculée {p0:.4f} vs notebook 05 {p_test[example]:.4f}")

In [ ]:
cases = {}
cancer_idx, benign_idx = np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)
tp = cancer_idx[p_test[cancer_idx] >= T_MAIN]
fn = cancer_idx[p_test[cancer_idx] < T_MAIN]
fp = benign_idx[p_test[benign_idx] >= T_MAIN]
tn = benign_idx[p_test[benign_idx] < T_MAIN]
if len(tp): cases["vrai positif"] = int(tp[np.argmax(p_test[tp])])
if len(fn): cases["faux négatif"] = int(fn[np.argmin(p_test[fn])])
if len(fp): cases["faux positif"] = int(fp[np.argmax(p_test[fp])])
if len(tn): cases["vrai négatif"] = int(tn[np.argmin(p_test[tn])])

fig, axes = plt.subplots(len(cases), 2, figsize=(7, 3.4 * len(cases)), squeeze=False)
for r, (kind, i) in enumerate(cases.items()):
    show_occlusion(axes[r, 0], axes[r, 1], i, f"{kind} — {test.loc[i, 'diagnostic']} ({test.loc[i, 'img_id']})")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_cartes_occlusion.png", dpi=130)
plt.show()

## 4. Fidélité des cartes : test de suppression

Pour les cancers du test les mieux reconnus, on masque les 25 % de zones jugées les plus importantes par la carte, puis 25 % de zones tirées au hasard (3 tirages). Si la carte est fidèle, la probabilité doit baisser **nettement plus** dans le premier cas (test de Wilcoxon apparié, unilatéral).

In [ ]:
from scipy.stats import wilcoxon

N_FIDELITY, K_TOP, N_RANDOM = 40, GRID * GRID // 4, 3
rng = np.random.default_rng(SEED)


def masked(x, cells):
    v = x.clone()
    for c in cells:
        si, sj = cell_slices(c // GRID, c % GRID, x.shape[1])
        v[:, si, sj] = 0.0
    return v


fidelity_cases = cancer_idx[np.argsort(-p_test[cancer_idx])][:N_FIDELITY]
rows, t0 = [], time.time()
for n, i in enumerate(fidelity_cases):
    x, p0, heat = occlusion_map(int(i))
    top = np.argsort(-heat.ravel())[:K_TOP]
    rand = [rng.choice(GRID * GRID, K_TOP, replace=False) for _ in range(N_RANDOM)]
    p = proba_of_tensors(int(i), [masked(x, top)] + [masked(x, r) for r in rand])
    rows.append({"img_id": test.loc[i, "img_id"], "p": p0, "baisse (zones désignées)": p0 - p[0],
                 "baisse (zones au hasard)": p0 - p[1:].mean()})
    if n % 10 == 0:
        print(f"{n + 1}/{len(fidelity_cases)} — {time.time() - t0:.0f} s")
fidelity = pd.DataFrame(rows)
stat = wilcoxon(fidelity["baisse (zones désignées)"], fidelity["baisse (zones au hasard)"], alternative="greater")
print(f"Baisse moyenne : zones désignées {fidelity['baisse (zones désignées)'].mean():.3f} | "
      f"hasard {fidelity['baisse (zones au hasard)'].mean():.3f} | Wilcoxon p = {stat.pvalue:.2g}")
fidelity.to_csv(REPORT_DIR / "06_fidelite_occlusion.csv", index=False)

## 5. Raccourci possible : marques d'encre chirurgicale

1. **Détection automatique** : pixels bleus à violets saturés (teinte 200–300°, saturation ≥ 0,35). Une image est marquée si plus de 0,2 % de ses pixels sont de l'encre. *Heuristique* : vérifiez la galerie ; vous pouvez corriger à la main en créant `data/processed/ink_manual.csv` (colonnes `img_id`, `ink` = 0/1), qui est alors prioritaire.
2. **Lien avec le diagnostic** : si l'encre est bien plus fréquente sur les cancers, c'est un raccourci disponible.
3. **Performance** avec et sans encre, et **attention** : part de l'importance des cartes d'occlusion tombant sur les zones d'encre, comparée à la part de surface qu'elles occupent (rapport > 1,5 : le modèle regarde l'encre).

In [ ]:
INK_MIN_FRACTION = 0.002


def ink_mask(rgb):
    hsv = rgb_to_hsv(np.clip(rgb, 0, 1))
    h, s, v = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]
    return (h >= 200) & (h <= 300) & (s >= 0.35) & (v >= 0.15) & (v <= 0.95)


def ink_fraction(path, max_side=256):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_side, max_side))
        return float(ink_mask(np.asarray(im, dtype=np.float32) / 255).mean())


t0 = time.time()
ink = pd.DataFrame({"img_id": df["img_id"], "fraction_encre": [ink_fraction(ROOT / p) for p in df["img_path"]]})
ink["encre"] = (ink["fraction_encre"] > INK_MIN_FRACTION).astype(int)
manual_path = PROCESSED_DIR / "ink_manual.csv"
if manual_path.exists():
    manual = pd.read_csv(manual_path).set_index("img_id")["ink"]
    ink.loc[ink["img_id"].isin(manual.index), "encre"] = ink["img_id"].map(manual).dropna().astype(int)
    print("Corrections manuelles appliquées :", len(manual))
ink = ink.merge(df[["img_id", "diagnostic", "label", "split"]], on="img_id")
print(f"Détection terminée en {time.time() - t0:.0f} s — images marquées : {ink['encre'].sum()} / {len(ink)}")
ink.groupby("diagnostic")["encre"].mean().mul(100).round(1).rename("% d'images avec encre")

In [ ]:
top_ink = ink.sort_values("fraction_encre", ascending=False)
gallery = pd.concat([top_ink.head(12), top_ink[top_ink["encre"] == 0].head(6)])
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for ax, (_, r) in zip(axes.ravel(), gallery.iterrows()):
    with Image.open(ROOT / df.set_index("img_id").loc[r["img_id"], "img_path"]) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(f"{r['diagnostic']} | encre {r['fraction_encre'] * 100:.1f} % | {'marquée' if r['encre'] else 'non'}", fontsize=8)
    ax.axis("off")
plt.suptitle("Vérification visuelle de la détection d'encre (12 plus fortes + 6 juste sous le seuil)")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_galerie_encre.png", dpi=110)
plt.show()

if "skin_marker_ink" in LABELS_FR:
    scores = pd.read_csv(PROCESSED_DIR / "concept_scores.csv").set_index("img_id").loc[ink["img_id"], "skin_marker_ink"]
    if ink["encre"].nunique() == 2:
        print(f"Concordance concept DermLIP « encre » / détection couleur : AUC {roc_auc_score(ink['encre'], scores):.3f}")

In [ ]:
ink_test = ink.set_index("img_id").loc[test["img_id"], "encre"].to_numpy()
rows = []
for name, m in [("avec encre", ink_test == 1), ("sans encre", ink_test == 0)]:
    if m.sum() and len(set(y_test[m])) == 2:
        r = grouped_bootstrap(y_test[m], p_test[m], groups_test[m])
        rows.append({"groupe": name, "images": int(m.sum()), "% cancer": round(y_test[m].mean() * 100, 1),
                     "AUC": round(r["AUC"], 3), "IC95": f"{r['IC95 bas']:.3f} – {r['IC95 haut']:.3f}"})
    else:
        rows.append({"groupe": name, "images": int(m.sum()), "% cancer": None, "AUC": None, "IC95": "trop peu d'images"})
display(pd.DataFrame(rows))

N_INK_MAPS = 20
attention = []
for i in np.flatnonzero(ink_test == 1)[:N_INK_MAPS]:
    x, p0, heat = occlusion_map(int(i))
    mask = ink_mask(to_rgb(x))
    step = x.shape[1] // GRID
    cover = mask[:step * GRID, :step * GRID].reshape(GRID, step, GRID, step).mean(axis=(1, 3))
    positive = np.clip(heat, 0, None)
    if positive.sum() > 0 and cover.sum() > 0:
        attention.append({"img_id": test.loc[i, "img_id"],
                          "part de surface encre": cover.mean(),
                          "part d'importance sur l'encre": float((positive * cover).sum() / positive.sum())})
attention = pd.DataFrame(attention)
if len(attention):
    ratio = attention["part d'importance sur l'encre"].mean() / attention["part de surface encre"].mean()
    print(f"Rapport importance / surface sur l'encre : {ratio:.2f} "
          f"({'⚠ le modèle semble regarder l encre' if ratio > 1.5 else 'pas de sur-attention marquée'})")
else:
    ratio = None
    print("Pas d'image de test avec encre exploitable pour l'analyse d'attention.")

## 6. Rapport texte par gabarit

Le texte est **construit à partir des sorties déjà calculées** (probabilité, seuil, contributions, carte d'occlusion). Un modèle de langage pourrait le reformuler, mais ne reçoit jamais l'image et ne peut pas modifier la décision.

In [ ]:
def value_fr(col, value):
    if pd.isna(value):
        return "non renseigné"
    if col == "age":
        return f"{value:.0f} ans"
    if col == "region":
        return str(value).lower()
    return "oui" if value >= 0.5 else "non"


def location_fr(heat):
    i, j = np.unravel_index(np.argmax(heat), heat.shape)
    vert = ["en haut", "au centre", "en bas"][min(2, i * 3 // GRID)]
    horiz = ["à gauche", "au centre", "à droite"][min(2, j * 3 // GRID)]
    return "au centre de l'image" if vert == horiz == "au centre" else f"{vert} {horiz} de l'image"


def generate_report(i_test):
    row, p = test.loc[i_test], p_test[i_test]
    contrib = local_contributions(X_test.iloc[[i_test]])

    def phrase(r):
        if r["type"] == "clinique":
            return f"{r['élément']} : {value_fr(r['clé'], row[r['clé']])} ({r['contribution']:+.2f})"
        return f"{r['élément']} ({r['contribution']:+.2f})"

    plus = [phrase(r) for _, r in contrib[contrib["contribution"] > 0.05].head(3).iterrows()]
    minus = [phrase(r) for _, r in contrib[contrib["contribution"] < -0.05].head(2).iterrows()]
    decision = ("suspicion de cancer : avis dermatologique recommandé" if p >= T_MAIN
                else "pas de signe fort de cancer selon le modèle : surveillance selon avis médical")
    _, _, heat = occlusion_map(i_test)
    lines = [
        f"Image {row['img_id']} — probabilité estimée de cancer : {p:.2f} (seuil de triage {T_MAIN:.2f}).",
        f"Conclusion du modèle : {decision}.",
        "Éléments en faveur : " + ("; ".join(plus) if plus else "aucun élément marquant") + ".",
        "Éléments en défaveur : " + ("; ".join(minus) if minus else "aucun élément marquant") + ".",
        f"Zone de l'image la plus déterminante : {location_fr(heat)}.",
        "Avertissement : outil d'aide à la décision, il ne remplace ni l'examen clinique ni la biopsie.",
    ]
    return "\n".join(lines)


reports = []
for kind, i in cases.items():
    text = generate_report(i)
    reports.append(f"### {kind.capitalize()} (diagnostic réel : {test.loc[i, 'diagnostic']})\n\n{text}\n")
    print(f"--- {kind} ---\n{text}\n")
(REPORT_DIR / "06_rapports_exemples.md").write_text("\n".join(reports), encoding="utf8")

## 7. Analyse des erreurs (seuil de triage)

In [ ]:
pred = (p_test >= T_MAIN).astype(int)
err = test.assign(proba=p_test, pred=pred, encre=ink_test)
by_diag = err.groupby("diagnostic").apply(lambda g: pd.Series({
    "images": len(g),
    "classés cancer (%)": round(g["pred"].mean() * 100, 1),
    "type d'erreur": "faux négatifs" if g["label"].iloc[0] == 1 else "faux positifs",
    "erreurs (%)": round(((g["pred"] != g["label"]).mean()) * 100, 1)}), include_groups=False)
display(by_diag)
by_region = err.groupby("region").apply(lambda g: pd.Series({
    "images": len(g), "erreurs (%)": round(((g["pred"] != g["label"]).mean()) * 100, 1)}), include_groups=False)
display(by_region.sort_values("erreurs (%)", ascending=False).head(8))

worst = pd.concat([err[(err["label"] == 1) & (err["pred"] == 0)].nsmallest(4, "proba"),
                   err[(err["label"] == 0) & (err["pred"] == 1)].nlargest(4, "proba")])
if len(worst):
    fig, axes = plt.subplots(1, len(worst), figsize=(2.6 * len(worst), 3.2), squeeze=False)
    for ax, (_, r) in zip(axes[0], worst.iterrows()):
        with Image.open(ROOT / r["img_path"]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"{'FN' if r['label'] else 'FP'} {r['diagnostic']}\np = {r['proba']:.2f}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "06_erreurs.png", dpi=120)
    plt.show()

## 8. Analyse par phototype (Fitzpatrick)

Le phototype n'est renseigné que pour une partie des patients et les peaux foncées (IV–VI) sont très rares dans PAD-UFES-20 : ces chiffres sont **indicatifs** et constituent surtout une limite à signaler.

In [ ]:
fitz = test["fitspatrick"]
rows = []
for name, m in [("I–II (claire)", fitz.isin([1, 2]).to_numpy()), ("III–VI", (fitz >= 3).to_numpy()),
                ("non renseigné", fitz.isna().to_numpy())]:
    row = {"phototype": name, "images": int(m.sum()), "% cancer": round(y_test[m].mean() * 100, 1) if m.sum() else None}
    if m.sum() >= 20 and len(set(y_test[m])) == 2:
        r = grouped_bootstrap(y_test[m], p_test[m], groups_test[m])
        row.update({"AUC": round(r["AUC"], 3), "IC95": f"{r['IC95 bas']:.3f} – {r['IC95 haut']:.3f}"})
    rows.append(row)
pd.DataFrame(rows)

## 9. Synthèse

In [ ]:
synthesis = {
    "modele": bundle["nom"],
    "importance_modalites": modalities.set_index("élément")["baisse d'AUC"].round(4).to_dict(),
    "fidelite_occlusion": {"baisse_zones_designees": round(float(fidelity["baisse (zones désignées)"].mean()), 4),
                           "baisse_zones_hasard": round(float(fidelity["baisse (zones au hasard)"].mean()), 4),
                           "wilcoxon_p": float(stat.pvalue)},
    "encre": {"images_marquees": int(ink["encre"].sum()),
              "pourcentage_par_diagnostic": ink.groupby("diagnostic")["encre"].mean().mul(100).round(1).to_dict(),
              "rapport_importance_surface": None if ratio is None else round(float(ratio), 3)},
}
(REPORT_DIR / "06_synthese_xai.json").write_text(json.dumps(synthesis, indent=2, ensure_ascii=False), encoding="utf8")
print(json.dumps(synthesis, indent=2, ensure_ascii=False))

## Conclusion (à compléter après exécution)

- **Modalités** : laquelle porte la décision (section 1) — cohérent avec les poids de la porte (notebook 05) si le modèle à porte est retenu.
- **Fidélité** : la baisse « zones désignées » doit être nettement supérieure à la baisse « hasard » (p < 0,05) ; sinon les cartes ne doivent pas être présentées comme des explications fiables.
- **Encre** : fréquence par diagnostic, performance avec / sans encre, rapport d'attention. Un rapport > 1,5 ou une AUC très supérieure sur les images avec encre signale un raccourci à corriger (par exemple masquer l'encre ou exclure ces images à l'entraînement).
- **Erreurs** : diagnostics les plus manqués (mélanome ?) et faux positifs typiques (kératoses ?).
- **Limites** : phototypes foncés presque absents, mélanomes peu nombreux, explications locales approximatives pour un modèle non linéaire.